<a href="https://colab.research.google.com/github/mjmj002/Sparta/blob/main/Python/Project/02_%EC%8B%9C%EA%B0%81%ED%99%94_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3~4 | 시각화
> **Team Sparta | 주식/코인 모니터링 웹앱 프로젝트**

---

## 이번 노트북에서 만들 것
1. plotly로 인터랙티브 라인 차트 만들기
2. 거래량 막대 차트 추가하기
3. 이동평균선 레이어 추가하기
4. 최종 `make_chart()` 함수로 합치기

> **이 노트북은 01번 노트북의 함수들이 필요합니다.**  
> 아래 셀에서 01번의 함수를 다시 붙여넣기 하거나 import 하세요.

In [1]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 01번 노트북 함수 재사용 — 아래에 붙여넣기
# (또는 01번 노트북 커널이 살아있으면 %run 01_데이터수집_실습.ipynb 사용)

def get_stock_history(ticker_symbol, period='1달'):
    period_map = {'1주': '1wk', '1달': '1mo', '3달': '3mo', '1년': '1y'}
    yf_period = period_map.get(period, '1mo')
    try:
        ticker = yf.Ticker(ticker_symbol)
        df = ticker.history(period=yf_period)
        if df.empty:
            return None
        return df[['Open', 'High', 'Low', 'Close', 'Volume']]
    except Exception as e:
        print(f'[오류] {e}')
        return None

def calculate_moving_average(df, periods=[5, 20]):
    result = df.copy()
    for period in periods:
        result[f'MA{period}'] = result['Close'].rolling(window=period).mean()
    return result

print('준비 완료!')

준비 완료!


---
## plotly 탐색
본격적인 함수 작성 전에, plotly의 기본 사용법을 확인합니다.

In [7]:
# plotly 가장 기본 라인 차트 — 구조 파악용
import plotly.graph_objects as go

df = get_stock_history('NVDA', '1달')

fig = go.Figure()

# Scatter trace: 점/선 차트
fig.add_trace(go.Scatter(
    x=df.index,           # x축: 날짜
    y=df['Close'],         # y축: 종가
    mode='lines',          # 'lines', 'markers', 'lines+markers'
    name='종가',
    line=dict(color='orange', width=2)
))

fig.update_layout(title='NVDA 종가 (기본 차트)', height=400)
fig.show()

In [8]:
# subplot: 차트를 위아래로 나누는 방법
from plotly.subplots import make_subplots

# rows=2: 2개 행, row_heights: 각 행의 높이 비율
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],   # 상단 70%, 하단 30%
    shared_xaxes=True,         # x축 공유 (줌할 때 같이 움직임)
    vertical_spacing=0.05
)

# 상단(row=1): 종가
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='종가'), row=1, col=1)

# 하단(row=2): 거래량
fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='거래량'), row=2, col=1)

fig.update_layout(title='종가 + 거래량 서브플롯', height=500)
fig.show()

---
## 함수 5 | `make_price_chart()` — 가격 라인 차트

### 요구사항
- 입력: DataFrame (OHLCV), 티커 심볼, 기간 문자열, 이동평균 표시 여부
- 출력: plotly Figure 객체
- 종가 라인 + 이동평균선(MA5, MA20) 포함
- 상승/하락 색상: 종가 > 시가 → 빨강, 아니면 파랑

In [39]:
def make_price_chart(df, ticker_symbol, period='1달', show_ma=True):
    """
    종가 라인 차트와 이동평균선을 그립니다.

    Args:
        df (pd.DataFrame): OHLCV 데이터
        ticker_symbol (str): 티커 심볼 (차트 제목용)
        period (str): 기간 문자열 (차트 제목용)
        show_ma (bool): 이동평균선 표시 여부

    Returns:
        go.Figure: plotly 차트 객체
    """
    # 이동평균 계산
    if show_ma:
        df = calculate_moving_average(df, periods=[5, 20])

    # Figure 생성
    # 힌트: go.Figure()
    fig = go.Figure()

    # # 종가 라인 추가
    # # 힌트: fig.add_trace(go.Scatter(x=___, y=___, mode='lines', name='종가', ...))
    # fig.add_trace(go.Scatter(
    #     x=df.index,          # 힌트: df.index (날짜가 인덱스)
    #     y=df['Close'],        # 힌트: 'Close'
    #     mode='lines',
    #     name='종가',
    #     line=dict(color='#2196F3', width=2)
    # ))

    # 상승/하락 여부에 따른 색상 리스트 생성
    # 종가(Close) > 시가(Open) 면 빨강(#FF1744), 아니면 파랑(#2979FF)
    colors = ['#FF1744'
              if row['Close'] > row['Open']
              else '#2979FF' for _, row in df.iterrows()]

    # 종가 라인 추가 (Scatter)
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['Close'],
        mode='lines+markers',     # 상승/하락 색상을 점(markers)에 적용하면 더 직관적입니다
        name='종가',
        line=dict(color='#757575', width=1.5), # 선은 중립적인 회색
        marker=dict(color=colors, size=5),     # 점에 상승(빨강)/하락(파랑) 적용
    ))

    # 시가 라인 추가
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['Open'],
        mode='lines',
        name='시가',
        line=dict(color='#BDBDBD', width=1, dash='dot'), # 시가는 연한 점선으로 표시
        hovertemplate='시가: %{y:,.0f}원<extra></extra>'
    ))


    # 이동평균선 추가 (show_ma=True일 때만)
    if show_ma:
        ma_colors = {'MA5': '#FF9800', 'MA20': '#9C27B0'}  # 5일: 주황, 20일: 보라

        for ma_col, color in ma_colors.items():
            # 힌트: fig.add_trace(go.Scatter(x=df.index, y=df[ma_col], ...))
            fig.add_trace(go.Scatter(
                x=df.index,
                y=df[ma_col],          # 힌트: ma_col
                mode='lines',
                name=ma_col,
                line=dict(color=color, width=1.5, dash='dot')  # 점선
            ))

    # 레이아웃 설정
    fig.update_layout(
        title=f'{ticker_symbol} 주가 차트 ({period})',
        xaxis_title='날짜',
        yaxis_title='가격',
        height=400,          # 힌트: 400 정도
        hovermode='x unified',
        showlegend=True,
        plot_bgcolor='white',
        paper_bgcolor='white',
    )

    # (추가) x축 눈금선(Grid)
    fig.update_xaxes(showgrid=True, gridwidth=0.5, griddash='dot', gridcolor='LightGrey')
    fig.update_yaxes(showgrid=True, gridwidth=0.5, griddash='dot', gridcolor='LightGrey')

    # Y축(가로선) 비활성화
    fig.update_yaxes(
      showgrid=False                        # 가로선을 끕니다
    )

    return fig

In [40]:
# 테스트
df = get_stock_history('NVDA', '3달')
fig = make_price_chart(df, 'NVDA', '3달', show_ma=True)
fig.show()

---
## 함수 6 | `make_combined_chart()` — 가격 + 거래량 통합 차트

### 요구사항
- 가격 차트(상단 70%) + 거래량 차트(하단 30%) 구성
- x축 공유 (하나를 줌하면 다른 것도 같이 줌)
- 거래량 바차트 색상: 상승일 빨강, 하락일 파랑

In [53]:
def make_combined_chart(df, ticker_symbol, period='1달', show_ma=True):
    """
    가격 차트(상단)와 거래량 차트(하단)를 합친 통합 차트를 생성합니다.

    Args:
        df (pd.DataFrame): OHLCV 데이터
        ticker_symbol (str): 티커 심볼
        period (str): 기간 문자열
        show_ma (bool): 이동평균선 표시 여부

    Returns:
        go.Figure: 통합 차트 객체
    """
    if show_ma:
        df = calculate_moving_average(df, periods=[5, 20])

    # subplot 생성 (위 탐색 코드 참고)
    # 힌트: make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], shared_xaxes=True, vertical_spacing=0.05)
    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.7, 0.3],    # 힌트: [0.7, 0.3]
        shared_xaxes=True,
        vertical_spacing=0.05
    )

    # 상단(row=1): 종가 라인
    fig.add_trace(
        go.Scatter(x=df.index, y=df['Close'], mode='lines',
                   name='종가', line=dict(color='#2196F3', width=2)),
        row=1, col=1    # 힌트: row=1
    )

    # 상단(row=1): 이동평균선
    if show_ma:
        for ma_col, color in [('MA5', '#FF9800'), ('MA20', '#9C27B0')]:
            fig.add_trace(
                go.Scatter(x=df.index, y=df[ma_col], mode='lines',
                           name=ma_col, line=dict(color=color, width=1.5, dash='dot')),
                row=1, col=1
            )

    # 하단(row=2): 거래량 바차트
    # 상승일(종가 >= 시가)이면 빨강, 하락일이면 파랑
    # 힌트: df['Close'] >= df['Open'] 조건으로 색상 리스트 만들기
    volume_colors = ['#E53935' if close >= open_ else '#1E88E5'
                     for close, open_ in zip(df['Close'], df['Open'])]  # 힌트: 'Close', 'Open'

    fig.add_trace(
        go.Bar(x=df.index, y=df['Volume'],    # 힌트: 'Volume'
               name='거래량', marker_color=volume_colors,
               showlegend=False),
        row=2, col=1    # 힌트: row=2
    )

    # 전체 레이아웃
    fig.update_layout(
        title=f'{ticker_symbol} 차트 ({period})',
        height=500,
        hovermode='x unified',
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    fig.update_yaxes(title_text='가격', row=1, col=1)
    fig.update_yaxes(title_text='거래량', row=2, col=1)

    return fig

In [54]:
# 테스트
df = get_stock_history('NVDA', '3달')
fig = make_combined_chart(df, 'NVDA', '3달', show_ma=True)
fig.show()

---
## 함수 7 | `make_comparison_chart()` — 여러 종목 비교 차트 (선택)

### 요구사항
- 여러 종목을 같은 차트에 표시
- 시작 시점을 100으로 **정규화**해서 비교 (주가 단위가 달라 그냥 비교 불가)
- 정규화: `(현재가 / 첫날 시작가) × 100`

In [57]:
def make_comparison_chart(tickers, period='1달'):
    """
    여러 종목을 정규화해서 한 차트에 비교합니다.

    Args:
        tickers (list): 티커 심볼 리스트 (예: ['AAPL', 'TSLA', 'GOOGL'])
        period (str): 조회 기간

    Returns:
        go.Figure: 비교 차트 객체
    """
    fig = go.Figure()

    for ticker_symbol in tickers:
        df = get_stock_history(ticker_symbol, period)
        if df is None:
            continue

        # 정규화: 첫 번째 종가를 기준(100)으로 변환
        # 힌트: df['Close'] / df['Close'].iloc[0] * 100
        normalized = df['Close'] / df['Close'].iloc[0] * 100

        fig.add_trace(go.Scatter(
            x=df.index,
            y=normalized,          # 힌트: normalized
            mode='lines',
            name=ticker_symbol
        ))

    # 기준선 100 추가 (점선)
    fig.add_hline(y=100, line_dash='dash', line_color='gray',
                  annotation_text='기준선(100)')

    fig.update_layout(
        title=f'종목 비교 ({period}, 시작가=100 기준)',
        yaxis_title='정규화 가격',
        height=400,
        hovermode='x unified'
    )
    return fig

In [58]:
# 테스트
fig = make_comparison_chart(['AAPL', 'TSLA', 'GOOGL'], '3달')
fig.show()

In [65]:
import plotly.graph_objects as go

def make_comparison_chart(tickers, period='1달'):
    fig = go.Figure()

    # 컬러 팔레트 (종목이 많아질 것에 대비)
    colors = ['#2E7D32', '#C62828', '#1565C0', '#EF6C00', '#6A1B9A', '#00838F']

    for i, ticker_symbol in enumerate(tickers):
        # 1. 데이터 가져오기 (외부 함수 호출)
        df = get_stock_history(ticker_symbol, period)
        if df is None or len(df) == 0:
            continue

        # 2. 정규화 및 수익률 계산
        start_price = df['Close'].iloc[0]
        normalized = (df['Close'] / start_price) * 100
        final_return = (df['Close'].iloc[-1] / start_price - 1) * 100 # 최종 수익률(%)

        # 3. 라인 추가
        color = colors[i % len(colors)]
        fig.add_trace(go.Scatter(
            x=df.index,
            y=normalized,
            mode='lines',
            name=f"<b>{ticker_symbol}</b> ({final_return:+.1f}%)", # 범례에 수익률 표시
            line=dict(width=1.5, color=color),
            hovertemplate=f'<b>{ticker_symbol}</b><br>기준대비: %{{y:.1f}}%<extra></extra>'
        ))

        # 4. 차트 끝에 종목명 라벨 추가 (가독성 향상)
        fig.add_annotation(
            x=df.index[-1],
            y=normalized.iloc[-1],
            text=f" {ticker_symbol}",
            showarrow=False,
            xanchor='left',
            font=dict(color=color, size=12)
        )

    # 5. 기준선(100) 및 레이아웃 설정
    fig.add_hline(y=100, line_dash='dash', line_color='#424242', line_width=1,
                  annotation_text='기준점(100)', annotation_position="bottom left")

    fig.update_layout(
        title=dict(
            text=f'<b>주요 종목 상대 수익률 비교</b> ({period} 기준)',
            x=0.5, xanchor='center', font=dict(size=18)
        ),
        xaxis=dict(showgrid=True, gridcolor='#F0F0F0', zeroline=False),
        yaxis=dict(
            title='수익률 지수 (시작=100)',
            showgrid=True, gridcolor='#F0F0F0',
            ticksuffix='%', # Y축 뒤에 % 추가
            zeroline=False
        ),
        height=500,
        margin=dict(l=50, r=80, t=80, b=50), # 우측 라벨 공간 확보
        hovermode='x unified',
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig

    # 테스트
fig = make_comparison_chart(['AAPL', 'NVDA', 'TSLA', 'GOOGL'], '3달')
fig.show()

---
## Step 3~4 마무리 체크

| 체크 | 항목 |
|:---:|------|
| ☐ | `make_price_chart()` 실행 시 라인 차트 표시 |
| ☐ | 이동평균선(MA5, MA20) 차트에 표시 |
| ☐ | `make_combined_chart()` 실행 시 가격+거래량 서브플롯 표시 |
| ☐ | 거래량 바차트 상승(빨강)/하락(파랑) 색상 구분 |
| ☐ | `make_comparison_chart()` 실행 시 여러 종목 비교 표시 |
| ☐ | x축 줌 시 두 차트가 함께 이동 |

### 다음 단계 `03_Streamlit앱_실습.ipynb` 으로 이동해서 웹앱으로 만들어보세요!